# Deep Learning (Fall 2026) — Project 1: Try 5 (`ultra_sub5m` + AutoAugment + Mixup)

This notebook implements **Try 5** of our 100% from-scratch Residual Network (`CustomResNet`) for the CIFAR-10 Kaggle competition under the strict **5,000,000 trainable parameter ceiling**:
- **`ultra_sub5m` Architecture (`4,977,834` Parameters — 99.56% of Budget):** Uses $N=4$ stages, $C = [64, 144, 208, 280]$, $B = [2, 2, 2, 2]$, $F = [3, 3, 3, 3]$, $K = [1, 1, 1, 1]$, and $P = 4$.
- **CIFAR-10 AutoAugment + Cutout:** Applies `T.AutoAugment(T.AutoAugmentPolicy.CIFAR10)` alongside random cropping, flipping, and patch erasing.
- **Mixup Regularization ($\alpha = 0.3$):** Blends pairs of training images and labels on 50% of batches to prevent long-epoch overfitting.
- **100% Training Data Utilization:** Trains on all 50,000 competition images (`train_labels.csv` + `train_images/`) for 220 epochs and exports `/kaggle/working/submission.csv` (`id,label`) with Horizontal-Flip Test-Time Augmentation (TTA).

In [ ]:
# Optional: uncomment if torchsummary is not installed and internet is enabled
# !pip install -q torchsummary

In [ ]:
import csv, glob, json, math, os, pickle, random, time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional, Callable, Union

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as T

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

def seed_everything(seed: int = 2026) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True

seed_everything(2026)

## 1. From-Scratch Residual Network Architecture ($N, B_i, C_i, F_i, K_i, P$)
Each `ResidualUnit` computes $\text{ReLU}(S(x) + F(x))$, and `CustomResNet` chains $N$ stages of $B_i$ residual blocks followed by $P \times P$ average pooling and a linear classification head.

In [ ]:
@dataclass
class ResNetSpec:
    name: str
    stem_channels: int
    channels: List[int]          # C_i: channels in Residual Layer i
    blocks_per_stage: List[int]  # B_i: residual blocks in Residual Layer i
    conv_kernels: List[int]      # F_i: conv kernel size in Residual Layer i
    skip_kernels: List[int]      # K_i: skip connection kernel size in Residual Layer i
    pool_size: int               # P:   average pool kernel size
    dropout_rate: float = 0.0
    num_classes: int = 10

    def to_dict(self) -> dict:
        return asdict(self)

class ResidualUnit(nn.Module):
    """Implements ReLU(S(x) + F(x)) from scratch."""
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1, conv_kernel: int = 3, skip_kernel: int = 1):
        super().__init__()
        conv_pad = conv_kernel // 2
        self.residual_branch = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=conv_kernel, stride=stride, padding=conv_pad, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=conv_kernel, stride=1, padding=conv_pad, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        if stride != 1 or in_channels != out_channels:
            skip_pad = skip_kernel // 2
            self.skip_connection = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=skip_kernel, stride=stride, padding=skip_pad, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.skip_connection = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.relu(self.skip_connection(x) + self.residual_branch(x), inplace=True)

class CustomResNet(nn.Module):
    """Configurable N-Stage Residual Network built from scratch for 32x32 CIFAR-10 images."""
    def __init__(self, spec: ResNetSpec):
        super().__init__()
        self.spec = spec
        self.current_channels = spec.stem_channels

        self.stem = nn.Sequential(
            nn.Conv2d(3, spec.stem_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(spec.stem_channels),
            nn.ReLU(inplace=True),
        )

        stages = []
        for idx in range(len(spec.channels)):
            stage_stride = 1 if idx == 0 else 2
            stages.append(self._build_stage(
                out_channels=spec.channels[idx],
                num_blocks=spec.blocks_per_stage[idx],
                first_stride=stage_stride,
                conv_kernel=spec.conv_kernels[idx],
                skip_kernel=spec.skip_kernels[idx],
            ))
        self.residual_stages = nn.Sequential(*stages)
        self.pool_size = spec.pool_size
        self.dropout = nn.Dropout(p=spec.dropout_rate) if spec.dropout_rate > 0 else nn.Identity()

        final_map_width = 32 // (2 ** (len(spec.channels) - 1))
        pooled_width = max(1, final_map_width // spec.pool_size)
        fc_input_dim = spec.channels[-1] * (pooled_width ** 2)
        self.classifier = nn.Linear(fc_input_dim, spec.num_classes)
        self._init_weights()

    def _build_stage(self, out_channels, num_blocks, first_stride, conv_kernel, skip_kernel):
        blocks = []
        for b_idx in range(num_blocks):
            stride = first_stride if b_idx == 0 else 1
            blocks.append(ResidualUnit(self.current_channels, out_channels, stride, conv_kernel, skip_kernel))
            self.current_channels = out_channels
        return nn.Sequential(*blocks)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.01); nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.residual_stages(x)
        x = F.avg_pool2d(x, kernel_size=self.pool_size)
        x = torch.flatten(x, start_dim=1)
        return self.classifier(self.dropout(x))

def count_trainable_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 2. Parameter Count Verification (`<= 5,000,000`) — Including `ultra_sub5m` (`4,977,834` Parameters)

In [ ]:
ARCHITECTURE_REGISTRY = {
    "baseline_4stage": ResNetSpec(
        name="baseline_4stage", stem_channels=64, channels=[64, 128, 192, 256],
        blocks_per_stage=[2, 2, 2, 2], conv_kernels=[3, 3, 3, 3], skip_kernels=[1, 1, 1, 1],
        pool_size=4, dropout_rate=0.0
    ),
    "wide_3stage": ResNetSpec(
        name="wide_3stage", stem_channels=64, channels=[64, 144, 264],
        blocks_per_stage=[3, 3, 3], conv_kernels=[3, 3, 3], skip_kernels=[1, 1, 1],
        pool_size=8, dropout_rate=0.05
    ),
    "skip3x3_4stage": ResNetSpec(
        name="skip3x3_4stage", stem_channels=64, channels=[64, 128, 192, 256],
        blocks_per_stage=[2, 2, 2, 2], conv_kernels=[3, 3, 3, 3], skip_kernels=[1, 3, 3, 3],
        pool_size=4, dropout_rate=0.05
    ),
    "champion_sub5m": ResNetSpec(
        name="champion_sub5m", stem_channels=64, channels=[64, 128, 208, 264],
        blocks_per_stage=[3, 3, 2, 2], conv_kernels=[3, 3, 3, 3], skip_kernels=[1, 1, 1, 1],
        pool_size=4, dropout_rate=0.1
    ),
    "ultra_sub5m": ResNetSpec(
        name="ultra_sub5m", stem_channels=64, channels=[64, 144, 208, 280],
        blocks_per_stage=[2, 2, 2, 2], conv_kernels=[3, 3, 3, 3], skip_kernels=[1, 1, 1, 1],
        pool_size=4, dropout_rate=0.0
    ),
}

rows = []
for name, spec in ARCHITECTURE_REGISTRY.items():
    net = CustomResNet(spec)
    params = count_trainable_parameters(net)
    assert params <= 5_000_000, f"{name} exceeds 5M parameters!"
    rows.append({
        "Architecture": name,
        "N (Stages)": len(spec.channels),
        "C_i (Channels)": str(spec.channels),
        "B_i (Blocks)": str(spec.blocks_per_stage),
        "F_i (Filters)": str(spec.conv_kernels),
        "K_i (Skip)": str(spec.skip_kernels),
        "P (Pool)": spec.pool_size,
        "Trainable Parameters": f"{params:,}",
        "% of 5M Budget": f"{100.0 * params / 5_000_000:.2f}%"
    })
display(pd.DataFrame(rows))

ultra_net = CustomResNet(ARCHITECTURE_REGISTRY["ultra_sub5m"]).to(DEVICE)
print(f"\nTry 5 Model (ultra_sub5m) Trainable Parameters: {count_trainable_parameters(ultra_net):,} (99.56% of 5,000,000 budget)")

## 3. Dataset Pipeline: CIFAR-10 AutoAugment + 100% Competition Data (50,000 Images)

In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2023, 0.1994, 0.2010)
CIFAR10_CLASSES = [f"class_{i}" for i in range(10)]
BATCH_SIZE = 128

# Upgraded data augmentation with official CIFAR-10 AutoAugment
train_transform = T.Compose([
    T.RandomCrop(32, padding=4, padding_mode="reflect"),
    T.RandomHorizontalFlip(p=0.5),
    T.AutoAugment(T.AutoAugmentPolicy.CIFAR10),
    T.ToTensor(),
    T.Normalize(CIFAR10_MEAN, CIFAR10_STD),
    T.RandomErasing(p=0.25, scale=(0.02, 0.2), value=0.0),
])

eval_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

class CIFARCSVDataset(Dataset):
    def __init__(self, data_source, img_dir, transform=None, is_test=False):
        self.df = data_source.reset_index(drop=True).copy() if isinstance(data_source, pd.DataFrame) else pd.read_csv(data_source)
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.is_test = is_test
        self.classes = CIFAR10_CLASSES

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        fname = str(row["filename"]).strip() if "filename" in row and pd.notna(row["filename"]) else (f"test_{int(row['id']):04d}.png" if self.is_test else f"train_{int(row['id']):05d}.png")
        image = Image.open(self.img_dir / fname).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        if self.is_test:
            return int(row["id"]), image
        return image, int(row["label"])

# Locate competition paths across /kaggle/input/, ./, or /content/
train_csv_matches = glob.glob("/kaggle/input/**/train_labels.csv", recursive=True) + glob.glob("./**/train_labels.csv", recursive=True) + glob.glob("/content/**/train_labels.csv", recursive=True)
train_png_matches = glob.glob("/kaggle/input/**/train_00000.png", recursive=True) + glob.glob("./**/train_00000.png", recursive=True) + glob.glob("/content/**/train_00000.png", recursive=True)

assert train_csv_matches, "train_labels.csv not found!"
assert train_png_matches, "train_00000.png not found!"

TRAIN_CSV_PATH = Path(train_csv_matches[0])
TRAIN_IMG_DIR = Path(train_png_matches[0]).parent

print(f"Kaggle Train CSV : {TRAIN_CSV_PATH}")
print(f"Kaggle Train Dir : {TRAIN_IMG_DIR}")

full_train_df = pd.read_csv(TRAIN_CSV_PATH)

# Train on 100% of data (all 50,000 images); use a 2,500 sample for checkpoint validation
train_set = CIFARCSVDataset(full_train_df, TRAIN_IMG_DIR, transform=train_transform, is_test=False)
val_sample_df = full_train_df.sample(n=2500, random_state=2026).reset_index(drop=True)
test_set = CIFARCSVDataset(val_sample_df, TRAIN_IMG_DIR, transform=eval_transform, is_test=False)

use_cuda = torch.cuda.is_available()
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=use_cuda)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2, pin_memory=use_cuda)

print(f"Loaded: Training on all {len(train_set):,} images with AutoAugment | Fast checkpoint validation on {len(test_set):,} images")

## 4. Training Engine with Mixup Regularization ($\alpha = 0.3$)

In [ ]:
@torch.no_grad()
def evaluate_network(model: nn.Module, loader: DataLoader, loss_fn: nn.Module, apply_tta: bool = False) -> Tuple[float, float]:
    model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        logits = model(images)
        if apply_tta:
            logits = 0.5 * (logits + model(torch.flip(images, dims=[3])))
        loss = loss_fn(logits, labels)
        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_samples += labels.size(0)
    return total_loss / total_samples, 100.0 * total_correct / total_samples

def mixup_data(x, y, alpha=0.3):
    """Generates mixed inputs, label pairs, and mixing coefficient lambda."""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def train_and_evaluate(spec: ResNetSpec, epochs: int = 220, peak_lr: float = 0.1, weight_decay: float = 5e-4, label_smoothing: float = 0.05, out_dir: Path = Path('./outputs')):
    seed_everything(2026)
    out_dir.mkdir(parents=True, exist_ok=True)
    model = CustomResNet(spec).to(DEVICE)
    param_count = count_trainable_parameters(model)
    assert param_count <= 5_000_000, f"Model exceeds 5M parameters: {param_count:,}"

    train_loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    eval_loss_fn = nn.CrossEntropyLoss()

    matrix_params = [p for p in model.parameters() if p.requires_grad and p.ndim >= 2]
    vector_params = [p for p in model.parameters() if p.requires_grad and p.ndim < 2]
    optimizer = optim.SGD([
        {"params": matrix_params, "weight_decay": weight_decay},
        {"params": vector_params, "weight_decay": 0.0},
    ], lr=peak_lr, momentum=0.9, nesterov=True)

    warmup = min(8, max(1, epochs // 10))
    scheduler = optim.lr_scheduler.LambdaLR(
        optimizer,
        lambda e: float(e + 1) / warmup if e < warmup else 0.5 * (1.0 + math.cos(math.pi * (e - warmup) / max(1, epochs - warmup)))
    )
    use_amp = (DEVICE.type == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    history = []
    best_acc = 0.0
    best_ckpt = out_dir / f"{spec.name}_best.pth"
    t0 = time.perf_counter()

    for epoch in range(1, epochs + 1):
        model.train()
        run_loss, run_total = 0.0, 0
        cur_lr = optimizer.param_groups[0]["lr"]

        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)

            # Apply Mixup on 50% of training batches to regularize deep training
            use_mixup = (np.random.rand() > 0.5)
            if use_mixup:
                images, y_a, y_b, lam = mixup_data(images, labels, alpha=0.3)

            if use_amp:
                with torch.autocast("cuda", dtype=torch.float16):
                    logits = model(images)
                    loss = mixup_criterion(train_loss_fn, logits, y_a, y_b, lam) if use_mixup else train_loss_fn(logits, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(images)
                loss = mixup_criterion(train_loss_fn, logits, y_a, y_b, lam) if use_mixup else train_loss_fn(logits, labels)
                loss.backward()
                optimizer.step()

            run_loss += loss.item() * labels.size(0)
            run_total += labels.size(0)

        scheduler.step()
        tr_loss = run_loss / run_total
        te_loss, te_acc = evaluate_network(model, test_loader, eval_loss_fn, apply_tta=False)

        if te_acc > best_acc:
            best_acc = te_acc
            torch.save({"model_state_dict": model.state_dict(), "spec": spec.to_dict(), "best_test_acc": best_acc, "trainable_parameters": param_count}, best_ckpt)

        history.append({"epoch": epoch, "lr": cur_lr, "train_loss": tr_loss, "test_loss": te_loss, "test_acc": te_acc, "best_test_acc": best_acc})
        if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
            print(f"[{spec.name}] Epoch {epoch:03d}/{epochs:03d} (LR={cur_lr:.4f}) | Train Loss: {tr_loss:.4f} | Val Loss: {te_loss:.4f} | Val Acc: {te_acc:.2f}% (Best: {best_acc:.2f}%)")

    elapsed = time.perf_counter() - t0
    ckpt = torch.load(best_ckpt, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    _, std_acc = evaluate_network(model, test_loader, eval_loss_fn, apply_tta=False)
    _, tta_acc = evaluate_network(model, test_loader, eval_loss_fn, apply_tta=True)

    print("\n" + "=" * 76)
    print(f"FINAL RESULTS — {spec.name}")
    print(f"  Total Trainable Parameters : {param_count:,} (<= 5,000,000 limit)")
    print(f"  Best Validation Acc (Std)  : {std_acc:.2f}%")
    print(f"  Best Validation Acc (TTA)  : {tta_acc:.2f}%")
    print(f"  Total Training Time        : {elapsed/60.0:.2f} min")
    print("=" * 76)
    return {"model": model, "spec": spec.to_dict(), "parameters": param_count, "best_test_acc": std_acc, "best_test_acc_tta": tta_acc, "history": history}

## 5. Train `ultra_sub5m` (`4,977,834` Parameters) & Plot Curves

In [ ]:
EPOCHS = 220
OUT_DIR = Path("./outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

champ_spec = ARCHITECTURE_REGISTRY["ultra_sub5m"]
champ_res = train_and_evaluate(
    champ_spec,
    epochs=EPOCHS,
    peak_lr=0.1,
    weight_decay=5e-4,
    label_smoothing=0.05,
    out_dir=OUT_DIR
)

# Plot Training Curves
history = champ_res["history"]
epochs_list = [h["epoch"] for h in history]
best_acc = champ_res["best_test_acc"]
params = champ_res["parameters"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
axes[0].plot(epochs_list, [h["train_loss"] for h in history], label="Train Loss", color="#1f77b4", lw=2)
axes[0].plot(epochs_list, [h["test_loss"] for h in history], label="Val Loss", color="#d62728", lw=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].set_title(f"Loss Curves ({params:,} params)")
axes[0].grid(True, linestyle="--", alpha=0.4)
axes[0].legend()

axes[1].plot(epochs_list, [h["test_acc"] for h in history], label=f"Val Accuracy (Best: {best_acc:.2f}%)", color="#2ca02c", lw=2)
axes[1].axhline(90.0, color="gray", linestyle=":", label="90% Target Baseline")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].set_title("Validation Accuracy Progression")
axes[1].grid(True, linestyle="--", alpha=0.4)
axes[1].legend()

fig.tight_layout()
fig.savefig(OUT_DIR / "ultra_sub5m_training_curves.png", dpi=160)
plt.show()

## 6. Generate Kaggle Competition `submission.csv` (`id,label`)

In [ ]:
test_csv_matches = glob.glob("/kaggle/input/**/test.csv", recursive=True) + glob.glob("./**/test.csv", recursive=True) + glob.glob("/content/**/test.csv", recursive=True)
test_png_matches = glob.glob("/kaggle/input/**/test_0000.png", recursive=True) + glob.glob("./**/test_0000.png", recursive=True) + glob.glob("/content/**/test_0000.png", recursive=True)

assert test_csv_matches, "Could not find test.csv! Check that the competition data is added."
assert test_png_matches, "Could not find test_0000.png! Check that the competition data is added."

TEST_CSV_PATH = Path(test_csv_matches[0])
TEST_IMG_DIR = Path(test_png_matches[0]).parent

print(f"Kaggle Test CSV : {TEST_CSV_PATH}")
print(f"Kaggle Test Dir : {TEST_IMG_DIR}")

kaggle_test_set = CIFARCSVDataset(TEST_CSV_PATH, TEST_IMG_DIR, transform=eval_transform, is_test=True)
use_cuda = torch.cuda.is_available()
kaggle_test_loader = DataLoader(kaggle_test_set, batch_size=256, shuffle=False, num_workers=2, pin_memory=use_cuda)

model_eval = champ_res["model"].eval()
all_ids, all_preds = [], []
with torch.no_grad():
    for batch_ids, images in kaggle_test_loader:
        images = images.to(DEVICE)
        logits = 0.5 * (model_eval(images) + model_eval(torch.flip(images, dims=[3])))
        preds = logits.argmax(dim=1).cpu().tolist()
        all_ids.extend([int(i) for i in batch_ids])
        all_preds.extend([int(p) for p in preds])

sub_df = pd.DataFrame({"id": all_ids, "label": all_preds})
assert list(sub_df.columns) == ["id", "label"], "Columns must be exactly: id,label"
assert len(sub_df) == len(kaggle_test_set), f"Expected {len(kaggle_test_set)} rows, got {len(sub_df)}"

sub_df.to_csv("submission.csv", index=False)
if Path("/kaggle/working").exists():
    sub_df.to_csv("/kaggle/working/submission.csv", index=False)

print(f"\nSUCCESS: Generated submission.csv with {len(sub_df)} predictions!")
print("\nPredicted Class Distribution (0-9):")
print(sub_df["label"].value_counts().sort_index().to_string())
print("\nFirst 10 rows:")
print(sub_df.head(10).to_string(index=False))